# Distributed Cluster Job Manager — Demo

Run this notebook on **FABRIC JupyterHub** (`jupyter.fabric-testbed.net`).

All logic lives in the `.py` modules. This notebook only:
1. Checks FABlib config
2. Provisions the cluster
3. Calls demo methods
4. Launches the UI
5. Tears down the slice

---
## Cell 1 — FABlib config check

In [ ]:
from fabrictestbed_extensions.fablib.fablib import FablibManager as fablib_manager

fablib = fablib_manager()
fablib.show_config()

---
## Cell 2 — Provision cluster

Creates **3 heterogeneous FABRIC VMs** (8 / 4 / 2 cores) so the scheduler's
quota decisions are non-trivial. Takes ~2-3 minutes.

> If the kernel restarted but the slice is still alive, replace
> `system.provision(fablib)` with `system.reconnect(fablib)`.

In [ ]:
from system import ClusterSystem

system = ClusterSystem()
fabric_slice = system.provision(fablib, site="GPN")

fabric_slice.show()
fabric_slice.list_nodes()

---
## Cell 3 — Inspect cluster node specs

In [ ]:
system.show_cluster()

---
## Demo A — Quota-based scheduling + distributed execution

Three jobs with different resource demands are submitted simultaneously.
The scheduler computes **weighted fair-share quotas** across all jobs,
decomposes large jobs into chunks that fill their quota across multiple nodes,
and runs all chunks in parallel via SSH.

Wall-clock time proves real distribution: `heavy_job` runs 3 chunks on
3 VMs simultaneously, finishing in ~20s instead of ~60s sequential.

In [ ]:
system.demo_quota_and_distribution()

---
## Demo B — Timeout detection

A job with `time_limit=15s` runs `sleep 999`. The monitor polls every 5s,
detects the exceeded deadline, and marks the job failed with a timeout error.

In [ ]:
system.demo_timeout(poll_interval=5, polls=8)

---
## Demo C — Node failure + task rescheduling

Jobs run across all nodes. After 10s, `worker-small` is marked unresponsive.
The monitor reschedules its tasks to healthy nodes and continues execution.

In [ ]:
system.demo_node_failure(fail_node="worker-small", failure_delay=10)

---
## Widget UI

Interactive interface: submit jobs, view quotas, monitor cluster, inspect jobs.

In [ ]:
system.show_ui()

---
## Teardown

In [ ]:
system.teardown(fablib)
print("Done — all FABRIC VMs released.")